# EmoExpress: End-to-End Pipeline and Streamlit Preparation

## Objective

This notebook integrates the complete EmoExpress workflow:

1. Validate the user's story.
2. Predict emotions using the selected production model.
3. Classify the primary and secondary topics.
4. Retrieve relevant guidance from the PDF knowledge base.
5. Generate a grounded empathetic response and recommendations.
6. Generate an encouraging caption and image prompt.
7. Generate a personalized supportive image.
8. Return all results in one structured output.

The completed pipeline will later be transferred into a Streamlit
application for interactive use.

Import Libraries

In [ ]:
from pathlib import Path
from datetime import datetime

import base64
import json
import os
import re
import time

import numpy as np
import pandas as pd
import torch

from PIL import Image as PILImage
from PIL import ImageDraw, ImageFont
from dotenv import load_dotenv
from IPython.display import Image as NotebookImage
from IPython.display import display

from openai import OpenAI

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


Configure project paths

In [ ]:
PROJECT_ROOT = Path.cwd().parent

MODEL_DIR = PROJECT_ROOT / "models"

ROBERTA_MODEL_PATH = (MODEL_DIR/ "roberta-goemotions-final")

KNOWLEDGE_BASE_DIR = (PROJECT_ROOT/ "knowledge_base")

VECTOR_STORE_DIR = (PROJECT_ROOT/ "vector_store"/ "chroma_db")

OUTPUT_DIR = PROJECT_ROOT / "outputs"

GENERATED_IMAGE_DIR = (OUTPUT_DIR/ "generated_images")

GENERATED_RESPONSE_DIR = (OUTPUT_DIR/ "generated_responses")

for directory in [GENERATED_IMAGE_DIR,GENERATED_RESPONSE_DIR]:
    directory.mkdir(parents=True,exist_ok=True)

TOPIC_MODEL_DIR = (MODEL_DIR / "topic_classifier"/ "distilbert_topic_classifier")

if not TOPIC_MODEL_DIR.exists():
    raise FileNotFoundError(f"Topic classifier not found: {TOPIC_MODEL_DIR}")

# print("Topic model directory:", TOPIC_MODEL_DIR)

# print("Project root:", PROJECT_ROOT)
# print("Model directory:", MODEL_DIR)
# print("Vector store:", VECTOR_STORE_DIR)

Load OpenAI API key

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY was not found.")

client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialized.")

**PART 1: Load the Emotion model**

Using Pretrained RoBERTa

In [ ]:
EMOTION_MODEL_NAME = ("SamLowe/roberta-base-go_emotions")

Load the RoBERTa tokenizer and model

In [ ]:
emotion_tokenizer = (AutoTokenizer.from_pretrained(EMOTION_MODEL_NAME))

emotion_model = (AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL_NAME))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

emotion_model.to(device)
emotion_model.eval()

print("Emotion model:", EMOTION_MODEL_NAME)
print("Device:", device)

Get the emotion labels

In [ ]:
id_to_label = emotion_model.config.id2label

emotion_labels = [id_to_label[index] for index in range(len(id_to_label))]

print("Number of emotion labels:", len(emotion_labels))
print(emotion_labels)

Create the emotion prediction function

In [ ]:
EMOTION_THRESHOLD = 0.50
MAX_EMOTIONS = 3

In [ ]:
def predict_emotions(user_story,
                    tokenizer=emotion_tokenizer,
                    model=emotion_model,
                    threshold=EMOTION_THRESHOLD,
                    maximum_emotions=MAX_EMOTIONS):
    """
    Predict the most relevant emotions from a user story.
    """

    if not isinstance(user_story, str):
        raise TypeError("user_story must be a string.")

    user_story = user_story.strip()

    if not user_story:
        raise ValueError("The user story cannot be empty.")

    encoded_input = tokenizer(user_story,return_tensors="pt",truncation=True,max_length=128,padding=True)

    encoded_input = {key: value.to(device) for key, value in encoded_input.items()}

    with torch.no_grad():
        output = model(**encoded_input)

    probabilities = torch.sigmoid(output.logits)[0].cpu().numpy()

    selected_indices = np.where(probabilities >= threshold)[0]

    if len(selected_indices) == 0:
        selected_indices = np.array([int(np.argmax(probabilities))])

    selected_indices = sorted(selected_indices,key=lambda index: probabilities[index],reverse=True)[:maximum_emotions]

    predictions = [{"emotion": emotion_labels[index],"score": float(probabilities[index])} for index in selected_indices]

    return predictions

In [ ]:
predict_emotions("I have another interview tomorrow and feel very nervous.")

**Part 2 — Topic classification**

Load the saved DistilBERT topic classifier

In [ ]:
topic_tokenizer = AutoTokenizer.from_pretrained(str(TOPIC_MODEL_DIR))

topic_model = (AutoModelForSequenceClassification.from_pretrained(str(TOPIC_MODEL_DIR)))

topic_model.to(device)
topic_model.eval()

print("Topic classifier loaded.")
print("Topic labels:", topic_model.config.id2label)

Normalization helper to ensure that every predicted topic matches a RAG folder.

In [ ]:
KNOWLEDGE_BASE_DIR = (PROJECT_ROOT/ "knowledge_base")

KNOWLEDGE_BASE_TOPICS = {directory.name for directory in KNOWLEDGE_BASE_DIR.iterdir() if directory.is_dir()}

print("Knowledge-base topics:",sorted(KNOWLEDGE_BASE_TOPICS))

In [ ]:
def normalize_topic_for_rag(predicted_topic: str) -> str:
    """
    Ensure the predicted topic matches an existing
    knowledge-base folder.
    """

    if predicted_topic in KNOWLEDGE_BASE_TOPICS:
        return predicted_topic

    return "general_support"

Predict the topic

In [ ]:
MAX_TOPIC_LENGTH = 128

def predict_topic(user_story: str) -> dict:
    """
    Predict one knowledge-base topic using the
    fine-tuned DistilBERT topic classifier.
    """

    if not isinstance(user_story, str):
        raise TypeError("user_story must be a string.")

    user_story = user_story.strip()

    if not user_story:
        raise ValueError("The user story cannot be empty.")

    encoded_input = topic_tokenizer(user_story,return_tensors="pt",truncation=True,max_length=MAX_TOPIC_LENGTH,padding=True)

    encoded_input = {key: value.to(device) for key, value in encoded_input.items()}

    with torch.no_grad():
        output = topic_model(**encoded_input)

    probabilities = torch.softmax(output.logits,dim=-1)[0]

    predicted_id = int(torch.argmax(probabilities).item())

    predicted_topic = (topic_model.config.id2label[predicted_id])

    predicted_topic = normalize_topic_for_rag(predicted_topic)

    confidence = float(
        probabilities[predicted_id].item())

    return {"topic": predicted_topic,
        "confidence": confidence,
        "model": "Fine-Tuned DistilBERT"}

**Part 3 — Load the RAG vector store**

*Inspect PDF files*

In [ ]:
# Find all pdf files
pdf_files = sorted(KNOWLEDGE_BASE_DIR.rglob("*.pdf"))

print("Number of PDF files:",len(pdf_files))

for pdf_path in pdf_files:
    print("-",pdf_path.relative_to(KNOWLEDGE_BASE_DIR),)

Initialize embeddings and Chroma

In [ ]:
EMBEDDING_MODEL = ("text-embedding-3-small")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL,api_key=OPENAI_API_KEY)

if not VECTOR_STORE_DIR.exists():
    raise FileNotFoundError("The Chroma database was not found. Run the RAG notebook first.")

vector_store = Chroma(persist_directory=str(VECTOR_STORE_DIR),embedding_function=embeddings)

print("Stored chunks:",vector_store._collection.count())

Create the retrieval-query function

In [ ]:
def build_retrieval_query(user_story,emotion_predictions,topic_result):
    """
    Combine the story, emotion predictions, and one
    predicted topic into a RAG retrieval query.
    """

    emotion_names = [prediction.get("emotion",prediction.get("label")) for prediction in emotion_predictions]

    emotion_names = [emotion for emotion in emotion_names if emotion]

    emotion_text = (", ".join(emotion_names) if emotion_names else "unspecified")

    predicted_topic = topic_result.get(     "topic","general_support")

    return f"""
User situation:
{user_story}

Detected emotions:
{emotion_text}

Predicted topic:
{predicted_topic}

Retrieve safe, practical, non-diagnostic guidance,
coping strategies, and constructive next steps
relevant to this situation.
""".strip()

Create the retrieval function

In [ ]:
def retrieve_documents(query,predicted_topic,k=5):
    primary_results = (vector_store.similarity_search_with_relevance_scores(query=query,k=k,filter={"topic": predicted_topic}))

    if primary_results:
        return (primary_results,"predicted_topic")

    general_results = (vector_store.similarity_search_with_relevance_scores(query=query,k=k,filter={"topic": "general_support"}))

    if general_results:
        return (general_results,"general_support")

    broad_results = (vector_store.similarity_search_with_relevance_scores(query=query,k=k))

    return (broad_results,"all_topics")

Format the retrieved context

In [ ]:
def format_retrieved_context(retrieved_results):
    context_sections = []
    sources = []

    for rank, (document,relevance_score) in enumerate(retrieved_results,start=1):
        title = document.metadata.get("document_title","Unknown document")

        page = document.metadata.get("page_number", "Unknown")

        topic = document.metadata.get("topic","Unknown")

        context_sections.append(f"""
SOURCE {rank}
Title: {title}
Page: {page}
Topic: {topic}

Passage:
{document.page_content}
""".strip())

        sources.append({
                "rank": rank,
                "title": title,
                "page": page,
                "topic": topic,
                "source_file": (document.metadata.get("source_file")),
                "relevance_score": float(relevance_score)})

    return ("\n\n".join(context_sections),sources)

**Part 4 — Grounded LLM generation**

Create the response system prompt

In [ ]:
RESPONSE_SYSTEM_PROMPT = """
You are EmoExpress, an empathetic emotional-support assistant.

Use the user story, predicted emotions, classified topic, and retrieved
PDF context to provide safe and practical support.

Rules:

1. Do not diagnose medical or mental-health conditions.
2. Do not recommend medication changes.
3. Do not claim to be a therapist or physician.
4. Do not invent citations or page numbers.
5. Use retrieved sources only when they support the recommendation.
6. Keep recommendations practical and manageable.
7. The caption must contain no more than 15 words.
8. The image prompt must represent emotional progress and realistic hope.
9. Do not include graphic distress, violence, self-harm, or medical treatment.
10. Return valid JSON only.

Required JSON format:

{
    "empathetic_response": "string",
    "recommendations": [
        {
            "recommendation": "string",
            "source_title": "string or null",
            "page": "integer or null"
        }
    ],
    "caption": "string",
    "image_prompt": "string",
    "retrieval_status": "grounded, partially_grounded, or not_grounded"
}
""".strip()

Generate the grounded response

In [ ]:
RESPONSE_MODEL = "gpt-4o-mini"

In [ ]:
def generate_grounded_response(user_story,emotion_predictions,topic_result,retrieved_context,retrieved_sources):
    prompt = f"""
USER STORY

{user_story}

EMOTION PREDICTIONS

{json.dumps(
    emotion_predictions,
    indent=2
)}

PREDICTED KNOWLEDGE-BASE TOPIC

{json.dumps(
    topic_result,
    indent=2
)}

RETRIEVED KNOWLEDGE

{retrieved_context}

AVAILABLE SOURCES

{json.dumps(
    retrieved_sources,
    indent=2
)}

Generate:

1. A short empathetic acknowledgment.
2. Two or three practical recommendations.
3. An encouraging caption.
4. A safe image-generation prompt.

Use citations only when supported by the provided sources.
""".strip()

    response = (
        client.chat.completions.create(
            model=RESPONSE_MODEL,
            temperature=0.3,
            response_format={
                "type": "json_object"
            },
            messages=[
                {
                    "role": "system",
                    "content": (
                        RESPONSE_SYSTEM_PROMPT
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
        )
    )

    return json.loads(
        response
        .choices[0]
        .message
        .content
    )

**Part 5 — Image generation**

Create the prompt-enhancement helper

In [ ]:
def prepare_image_prompt(image_prompt):
    return f"""
{image_prompt}

Create a polished and emotionally supportive digital illustration.

Requirements:

- Show symbolic emotional progress and realistic hope.
- Avoid depicting a specific identifiable person.
- Use a calm and balanced composition.
- Do not include text, logos, watermarks, or interface elements.
- Do not depict self-harm, violence, medication, or graphic distress.
""".strip()

Create a filename helper

In [ ]:
def create_safe_filename(caption):
    safe_caption = re.sub(r"[^a-zA-Z0-9]+","_",caption.lower()).strip("_")

    if not safe_caption:
        safe_caption = "emoexpress_image"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    return (
        f"{safe_caption[:50]}_"
        f"{timestamp}.png"
    )

Create the image-generation function

In [ ]:
IMAGE_MODEL = "gpt-image-1"

## Add the caption directly to the generated image

The image model generates the visual scene first. Pillow then overlays the exact
caption on the image so the final EmoExpress product is one picture containing
the personalized message.


In [ ]:
def load_caption_font(font_size):
    """Load a readable font with Windows and portable fallbacks."""

    font_candidates = [
        Path("C:/Windows/Fonts/arialbd.ttf"),
        Path("C:/Windows/Fonts/arial.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"),
    ]

    for font_path in font_candidates:
        if font_path.exists():
            return ImageFont.truetype(
                str(font_path),
                size=font_size,
            )

    return ImageFont.load_default()


def wrap_caption_text(
    draw,
    caption,
    font,
    maximum_width,
):
    """Wrap a caption according to its rendered width in pixels."""

    words = caption.split()

    if not words:
        return ""

    lines = []
    current_line = words[0]

    for word in words[1:]:
        candidate = f"{current_line} {word}"

        text_box = draw.textbbox(
            (0, 0),
            candidate,
            font=font,
        )

        text_width = (
            text_box[2]
            - text_box[0]
        )

        if text_width <= maximum_width:
            current_line = candidate
        else:
            lines.append(current_line)
            current_line = word

    lines.append(current_line)

    return "\n".join(lines)


def add_caption_to_image(
    image_path,
    caption,
    output_path=None,
):
    """
    Add a centered caption to a translucent banner near
    the bottom of an image.
    """

    if not isinstance(caption, str):
        raise TypeError(
            "caption must be a string."
        )

    caption = caption.strip()

    if not caption:
        raise ValueError(
            "caption cannot be empty."
        )

    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(
            f"Generated image was not found: {image_path}"
        )

    if output_path is None:
        output_path = image_path.with_name(
            f"{image_path.stem}_with_caption.png"
        )
    else:
        output_path = Path(output_path)

    image = PILImage.open(
        image_path
    ).convert("RGBA")

    image_width, image_height = image.size

    font_size = max(
        28,
        int(image_width * 0.045),
    )

    font = load_caption_font(
        font_size
    )

    overlay = PILImage.new(
        "RGBA",
        image.size,
        (0, 0, 0, 0),
    )

    draw = ImageDraw.Draw(
        overlay
    )

    horizontal_padding = int(
        image_width * 0.08
    )

    maximum_text_width = (
        image_width
        - 2 * horizontal_padding
    )

    wrapped_caption = wrap_caption_text(
        draw=draw,
        caption=caption,
        font=font,
        maximum_width=maximum_text_width,
    )

    text_box = draw.multiline_textbbox(
        (0, 0),
        wrapped_caption,
        font=font,
        spacing=10,
        align="center",
        stroke_width=2,
    )

    text_width = (
        text_box[2]
        - text_box[0]
    )

    text_height = (
        text_box[3]
        - text_box[1]
    )

    vertical_padding = max(
        24,
        int(image_height * 0.025),
    )

    banner_height = (
        text_height
        + 2 * vertical_padding
    )

    banner_top = (
        image_height
        - banner_height
        - int(image_height * 0.035)
    )

    banner_left = int(
        image_width * 0.045
    )

    banner_right = (
        image_width
        - banner_left
    )

    banner_bottom = (
        banner_top
        + banner_height
    )

    draw.rounded_rectangle(
        [
            banner_left,
            banner_top,
            banner_right,
            banner_bottom,
        ],
        radius=28,
        fill=(0, 0, 0, 175),
    )

    text_x = (
        image_width
        - text_width
    ) / 2

    text_y = (
        banner_top
        + vertical_padding
        - text_box[1]
    )

    draw.multiline_text(
        (text_x, text_y),
        wrapped_caption,
        font=font,
        fill=(255, 255, 255, 255),
        spacing=10,
        align="center",
        stroke_width=2,
        stroke_fill=(0, 0, 0, 220),
    )

    final_image = PILImage.alpha_composite(
        image,
        overlay,
    ).convert("RGB")

    final_image.save(
        output_path,
        format="PNG",
        quality=95,
    )

    return str(output_path)


In [ ]:
def generate_image(
    image_prompt,
    caption,
):
    final_prompt = prepare_image_prompt(
        image_prompt
    )

    response = client.images.generate(
        model=IMAGE_MODEL,
        prompt=final_prompt,
        size="1024x1024",
        quality="medium",
        n=1,
    )

    encoded_image = (
        response.data[0].b64_json
    )

    image_bytes = base64.b64decode(
        encoded_image
    )

    file_name = create_safe_filename(
        caption
    )

    original_image_path = (
        GENERATED_IMAGE_DIR
        / file_name
    )

    with open(
        original_image_path,
        "wb",
    ) as file:
        file.write(
            image_bytes
        )

    final_image_path = add_caption_to_image(
        image_path=original_image_path,
        caption=caption,
    )

    return {
        "status": "success",
        "path": final_image_path,
        "original_path": str(
            original_image_path
        ),
        "final_path": final_image_path,
        "caption": caption,
        "caption_embedded": True,
        "model": IMAGE_MODEL,
        "prompt": final_prompt,
    }


Add safe image error handling

In [ ]:
def generate_image_safely(
    image_prompt,
    caption,
):
    try:
        return generate_image(
            image_prompt=image_prompt,
            caption=caption,
        )

    except Exception as error:
        return {
            "status": "failed",
            "path": None,
            "original_path": None,
            "final_path": None,
            "caption": caption,
            "caption_embedded": False,
            "model": IMAGE_MODEL,
            "prompt": image_prompt,
            "error": str(error),
        }


**Part 6 — Complete pipeline**

Create the end-to-end function

In [ ]:
def run_emoexpress_pipeline(
    user_story,
    retrieval_k=5,
    generate_image_output=True,
):
    """
    Run the complete EmoExpress pipeline.
    """

    pipeline_start_time = time.perf_counter()

    if not isinstance(user_story, str):
        raise TypeError(
            "user_story must be a string."
        )

    user_story = user_story.strip()

    if len(user_story) < 5:
        raise ValueError(
            "Please enter a more detailed story."
        )

    # Emotion classification
    emotion_start = time.perf_counter()

    emotion_predictions = predict_emotions(
        user_story
    )

    emotion_time = (
        time.perf_counter()
        - emotion_start
    )

    # Topic classification
    topic_start = time.perf_counter()

    topic_result = predict_topic(
        user_story
    )

    topic_time = (
        time.perf_counter()
        - topic_start
    )

    predicted_topic = topic_result[
        "topic"
    ]

    # Retrieval query
    retrieval_query = build_retrieval_query(
        user_story=user_story,
        emotion_predictions=emotion_predictions,
        topic_result=topic_result,
    )

    # RAG retrieval
    retrieval_start = time.perf_counter()

    retrieved_results, strategy = (
        retrieve_documents(
            query=retrieval_query,
            predicted_topic=predicted_topic,
            k=retrieval_k,
        )
    )

    retrieval_time = (
        time.perf_counter()
        - retrieval_start
    )

    retrieved_context, sources = (
        format_retrieved_context(
            retrieved_results
        )
    )

    if not retrieved_context.strip():
        retrieved_context = (
            "No directly relevant knowledge-base "
            "passages were found."
        )
        sources = []

    # LLM response generation
    response_start = time.perf_counter()

    generated_response = (
        generate_grounded_response(
            user_story=user_story,
            emotion_predictions=emotion_predictions,
            topic_result=topic_result,
            retrieved_context=retrieved_context,
            retrieved_sources=sources,
        )
    )

    response_time = (
        time.perf_counter()
        - response_start
    )

    # Image generation
    image_result = {
        "status": "skipped",
        "path": None,
    }

    if generate_image_output:
        image_result = generate_image_safely(
            image_prompt=generated_response[
                "image_prompt"
            ],
            caption=generated_response[
                "caption"
            ],
        )

    total_time = (
        time.perf_counter()
        - pipeline_start_time
    )

    return {
        "user_story": user_story,
        "emotion_predictions": emotion_predictions,
        "topic_result": topic_result,
        "retrieval": {
            "query": retrieval_query,
            "predicted_topic": predicted_topic,
            "strategy": strategy,
            "sources": sources,
        },
        "generated_response": generated_response,
        "generated_image": image_result,
        "timing": {
            "emotion_seconds": emotion_time,
            "topic_seconds": topic_time,
            "retrieval_seconds": retrieval_time,
            "response_seconds": response_time,
            "total_seconds": total_time,
        },
    }

***Test the complete pipeline***

In [ ]:
sample_story = """
I have applied to many jobs and attended several interviews,
but I keep getting rejected. I am beginning to lose confidence
and wonder whether I am capable of succeeding.
""".strip()

In [ ]:
pipeline_result = (
    run_emoexpress_pipeline(
        user_story=sample_story,
        retrieval_k=5,
        generate_image_output=False,
    )
)

pipeline_result

In [ ]:
pipeline_result = (
    run_emoexpress_pipeline(
        user_story=sample_story,
        retrieval_k=5,
        generate_image_output=True,
    )
)

Display the result

In [ ]:
print("=" * 100)
print("DETECTED EMOTIONS")
print("=" * 100)

for prediction in pipeline_result["emotion_predictions"]:
    print(f"- {prediction['emotion']}: "
          f"{prediction['score']:.4f}")

print("\nPREDICTED TOPIC")

print(pipeline_result["topic_result"]["topic"])

print("\nTOPIC CONFIDENCE")

print(f"{pipeline_result['topic_result']['confidence']:.4f}")

print("\nTOPIC MODEL")

print(pipeline_result["topic_result"]["model"])

response = pipeline_result["generated_response"]

print("\n" + "=" * 100)
print("EMPATHETIC RESPONSE")
print("=" * 100)

print(response["empathetic_response"])

print("\nRECOMMENDATIONS")

for index, recommendation in enumerate(
    response["recommendations"],start=1):
    print(f"{index}. "
        f"{recommendation['recommendation']}")

    if recommendation.get("source_title"):
        print("   Source:", recommendation[ "source_title" ],
            "| Page:", recommendation.get( "page"))

print("\nCAPTION")

print(response["caption"])

print("\nIMAGE PROMPT")

print(response["image_prompt"])

print("\nSOURCES")

display(pd.DataFrame(pipeline_result["retrieval"]["sources"]))

print("\nTIMING")

display(pd.DataFrame([pipeline_result["timing"]]))

Display the image when available:

In [ ]:
image_result = pipeline_result[
    "generated_image"
]

final_image_path = image_result.get(
    "final_path",
    image_result.get("path"),
)

if final_image_path:
    print("FINAL EMOEXPRESS PRODUCT")
    print(
        "Caption embedded:",
        image_result.get(
            "caption_embedded",
            False,
        ),
    )

    display(
        NotebookImage(
            filename=final_image_path,
            width=700,
        )
    )
else:
    print(
        "Image status:",
        image_result.get(
            "status",
            "unknown",
        ),
    )

    if image_result.get("error"):
        print(
            "Image error:",
            image_result["error"],
        )


Save the complete result

In [ ]:
# Save JSON output
final_output_path = (GENERATED_RESPONSE_DIR/ "complete_pipeline_result.json")

with open(final_output_path,"w",encoding="utf-8") as file:
    json.dump(pipeline_result,file,ensure_ascii=False,indent=4)

print("Saved pipeline result:",final_output_path)

# Conclusion

This notebook successfully integrated all major EmoExpress components
into one end-to-end pipeline.

The final workflow accepted a user story, predicted multiple emotions,
classified one knowledge-base topic with the fine-tuned DistilBERT
classifier, retrieved topic-relevant PDF guidance, generated a grounded
empathetic response, and created a personalized supportive image.

The final image-processing stage now uses Pillow to overlay the exact
LLM-generated caption directly onto the generated image. As a result,
the final EmoExpress product is one complete visual containing both the
supportive scene and its personalized encouraging message.

The pipeline preserves both the original generated image and the final
captioned image. The captioned image path is returned as the main image
output so it can be displayed directly in the future Streamlit
application.
